# Diagnostic des données extraites

Avant d'écrire le pipeline de transformation, je regarde les données produites par l'extraction (étape 2) pour repérer les erreurs à corriger. Je reprends les types d'erreurs du cours : valeurs manquantes, erreurs de formatage, doublons, valeurs extrêmes.

Ce notebook sert seulement à explorer. Les corrections sont faites dans `src/transform.py`.

In [1]:
import json
from pathlib import Path

import pandas as pd
from PIL import Image

ROOT = Path("..")
files = sorted((ROOT / "data" / "extracted").glob("articles_*.json"))

dataframes = []
for file in files:
    with open(file, encoding="utf-8") as f:
        df_file = pd.DataFrame(json.load(f))
    df_file["fichier"] = file.name
    dataframes.append(df_file)

df = pd.concat(dataframes, ignore_index=True)
print(len(files), "fichiers,", df.shape[0], "articles,", df.shape[1], "colonnes")

11 fichiers, 613 articles, 14 colonnes


In [2]:
df.head(3)

,id,source,domaine,url,titre,texte,image_url,image_path,date_publication,langue,auteur,fiabilite_source,label,fichier
0,bd7ce5a86e742d2c77a2e1d57a26eb7a,newsdata,france24.com,https://www.france24.com/fr/%C3%A9co-tech/2026...,Nouveau test pour SpaceX : sa mégafusée Starsh...,L'entreprise d'Elon Musk doit tenter lundi d'a...,https://s.france24.com/media/display/3c5e7be4-...,data/images/bd7ce5a86e742d2c77a2e1d57a26eb7a.jpg,2026-09-27T22:46:11+00:00,fr,france 24,3118.0,NaN,articles_20260928_110726.json
1,d0f543bcb715efe80244bbb2244fc6b8,newsdata,laprovence.com,https://www.laprovence.com/article/france-mond...,SpaceX vise pour la première fois l'orbite ter...,Après son entrée retentissante en Bourse au mo...,https://pictures.laprovence.com/media/afp/a53e...,data/images/d0f543bcb715efe80244bbb2244fc6b8.jpg,2026-09-27T22:43:31+00:00,fr,afp import,6688.0,NaN,articles_20260928_110726.json
2,b09b49a8fb602543060c84a7c9b8e23d,newsdata,epochtimes.fr,https://www.epochtimes.fr/italie-un-rappeur-mo...,Italie : un rappeur moldave expulsé après avoi...,"Connu sous le nom de scène Bratan, Nicolae Nov...",https://images-fr.epochtimes.fr/uploads/2026/0...,data/images/b09b49a8fb602543060c84a7c9b8e23d.jpg,2026-09-27T22:37:00+00:00,fr,epochtimes.fr avec afp,10731.0,NaN,articles_20260928_110726.json


## 1. Valeurs manquantes

In [3]:
df.isnull().sum()

id                    0
source                0
domaine               0
url                   0
titre                 0
texte                 0
image_url             7
image_path            8
date_publication      0
langue                0
auteur              322
fiabilite_source    466
label               563
fichier               0
dtype: int64

In [4]:
# Répartition des valeurs manquantes par source
df.groupby("source")[["auteur", "fiabilite_source", "label", "image_path"]].apply(lambda g: g.isnull().sum())

,auteur,fiabilite_source,label,image_path
source,,,,
20minutes_fakeoff,0,100,100,0
franceinfo,138,138,138,0
legorafi,0,50,0,0
lemonde,80,80,80,0
lesdecodeurs,98,98,98,0
newsdata,6,0,147,8


- `image_path` manque pour les articles NewsData sans image : on les garde et on ajoutera une colonne `a_image`
- `auteur` manque souvent (les flux RSS ne le donnent pas toujours) : on le remplacera par "inconnu"
- `fiabilite_source` n'existe que pour NewsData : on le laisse vide pour les autres sources, le remplir serait inventer une valeur
- `label` n'existe que pour Le Gorafi ("satire") : pour les autres, personne n'a vérifié l'article, on mettra "non_verifie"

## 2. Types

In [5]:
df.dtypes

id                     str
source                 str
domaine                str
url                    str
titre                  str
texte                  str
image_url              str
image_path             str
date_publication       str
langue                 str
auteur                 str
fiabilite_source    object
label               object
fichier                str
dtype: object

In [6]:
print(df["date_publication"].head(3).tolist())
print(df["fiabilite_source"].dropna().head(3).tolist())

['2026-09-27T22:46:11+00:00', '2026-09-27T22:43:31+00:00', '2026-09-27T22:37:00+00:00']
[3118.0, 6688.0, 10731.0]


Les dates sont du texte et le rang de fiabilité est un décimal (`3118.0`) à cause des valeurs manquantes. Il faudra convertir les dates en datetime et le rang en entier.

## 3. Formatage des URL et doublons

In [7]:
# URL avec des paramètres après ? ou #
df["url"].str.contains(r"[?#]").groupby(df["source"]).sum()

source
20minutes_fakeoff    100
franceinfo           138
legorafi               0
lemonde                0
lesdecodeurs           0
newsdata               3
Name: url, dtype: int64

In [8]:
print(df.loc[df["source"] == "franceinfo", "url"].iloc[0])
print(df.loc[df["source"] == "20minutes_fakeoff", "url"].iloc[0])

https://www.franceinfo.fr/politique/rachida-dati/direct-proces-de-rachida-dati-suivez-le-dernier-jour-d-audience-et-les-requisitions-du-parquet-national-financier_8209622.html#xtor=RSS-3-[lestitres]
https://www.20minutes.fr/societe/4248812-20260925-prix-litteraire-peut-etre-decerne-livre-ecrit-aide-ia?at_medium=display&at_campaign=149


In [9]:
print("Doublons sur l'URL :", df.duplicated(subset="url").sum())
url_sans_suivi = df["url"].str.split("#").str[0].str.split("?").str[0]
print("Doublons sur l'URL sans paramètres :", url_sans_suivi.duplicated().sum())

Doublons sur l'URL : 391
Doublons sur l'URL sans paramètres : 392


- le même article revient à chaque extraction : il faut supprimer les doublons
- les URL de franceinfo et de 20 Minutes contiennent des paramètres de suivi (`#xtor=`, `?at_medium=`). Pour l'instant ils sont identiques d'une extraction à l'autre, donc ils ne changent pas le nombre de doublons. On les retire quand même : si un paramètre change (une autre campagne par exemple), le même article aurait deux URL différentes

## 4. Textes

In [10]:
df["texte"].str.len().describe()

count     613.000000
mean      327.551387
std       446.234173
min        76.000000
25%       159.000000
50%       209.000000
75%       263.000000
max      5292.000000
Name: texte, dtype: float64

In [11]:
fin_coupee = df["texte"].str.contains(r"(?:\.\.\.|…)$")
print("Textes qui finissent par ... :", fin_coupee.sum())
print(df.loc[fin_coupee, "texte"].iloc[0][-80:])

Textes qui finissent par ... : 30
jectifs ambitieux.Ce test, devant durer environ dix heures, doit voir l'étage...


In [12]:
print("Titres avec apostrophe typographique ’ :", df["titre"].str.contains("’").sum())
print("Titres avec apostrophe simple ' :", df["titre"].str.contains("'").sum())

Titres avec apostrophe typographique ’ : 213
Titres avec apostrophe simple ' : 127


- aucun texte vide ou très court
- certaines descriptions NewsData sont coupées et finissent par "..." : on retire ces points de suspension
- les deux types d'apostrophes sont mélangés (erreur d'irrégularité) : on garde seulement l'apostrophe simple

## 5. Images et association texte-image

In [13]:
# Même image pour plusieurs articles différents (après retrait des doublons)
sans_doublons = df[~url_sans_suivi.duplicated() & df["image_url"].notnull()]
partagees = sans_doublons[sans_doublons["image_url"].duplicated(keep=False)]
partagees[["domaine", "titre", "image_url"]]

,domaine,titre,image_url
5,lequipe.fr,Le ministre israélien des Affaires étrangères ...,https://medias.lequipe.fr/img-photo-png/-/1500...
8,lequipe.fr,"Avec 38 points, A'ja Wilson porte Las Vegas fa...",https://medias.lequipe.fr/img-photo-png/-/1500...
10,lequipe.fr,Paul Seixas doit-il s'en vouloir ?,https://medias.lequipe.fr/img-photo-png/-/1500...
18,lequipe.fr,"Un combat de rue, du suspense et la surprise M...",https://medias.lequipe.fr/img-photo-png/-/1500...
35,lemonde.fr,"EN DIRECT, guerre en Ukraine : les attaques ru...",https://img.lemde.fr/2026/09/28/845/0/6000/300...
36,lemonde.fr,Budget 2027 : le scénario prévoyant le gel de ...,https://img.lemde.fr/2026/01/06/250/0/3000/150...
162,lemonde.fr,"EN DIRECT, guerre en Ukraine : les attaques ru...",https://img.lemde.fr/2026/09/28/845/0/6000/300...
400,lemonde.fr,Quel est vraiment le niveau de vie des retrait...,https://img.lemde.fr/2026/01/06/250/0/3000/150...


In [14]:
avec_image = df[df["image_path"].notnull()]
print("Articles avec une image téléchargée :", len(avec_image), "/", len(df))

# Nom du fichier image = id de l'article ?
nom_ok = avec_image["image_path"].apply(lambda p: Path(p).stem) == avec_image["id"]
print("Images dont le nom ne correspond pas à l'id :", (~nom_ok).sum())

Articles avec une image téléchargée : 605 / 613
Images dont le nom ne correspond pas à l'id : 0


In [15]:
tailles = []
for path in avec_image["image_path"].unique():
    with Image.open(ROOT / path) as img:
        tailles.append({"largeur": img.width, "hauteur": img.height, "format": img.format})
tailles = pd.DataFrame(tailles)
print(tailles["format"].value_counts())
tailles.describe()

format
JPEG    194
PNG      20
Name: count, dtype: int64


,largeur,hauteur
count,214.000000,214.000000
mean,764.546729,455.247664
std,407.768754,266.142976
min,107.000000,56.000000
25%,432.000000,243.000000
50%,644.000000,322.000000
75%,1024.000000,600.000000
max,2560.000000,1440.000000


In [16]:
tailles.sort_values("largeur").head(3)

,largeur,hauteur,format
126,107,56,PNG
151,200,120,JPEG
153,200,120,JPEG


- toutes les images correspondent à l'id de leur article
- certaines images sont utilisées par plusieurs articles différents. Chez L'Équipe, c'est une image par défaut, chez Le Monde un direct dont l'URL a changé avec le titre : on les signale avec une colonne `image_partagee`
- quelques images sont très petites (valeurs extrêmes) : on les garde, leurs dimensions seront dans des colonnes

## 6. Langue et dates

In [17]:
print(df["langue"].value_counts())
dates = pd.to_datetime(df["date_publication"], utc=True)
print("Plus ancienne :", dates.min())
print("Plus récente :", dates.max())

langue
fr    613
Name: count, dtype: int64
Plus ancienne : 2026-09-16 14:00:00+00:00
Plus récente : 2026-09-28 15:29:37+00:00


Une seule langue et pas de date aberrante pour l'instant. Le pipeline vérifiera quand même ces deux points, car les prochaines extractions pourraient en contenir.

## Conclusion

Traitements à faire dans le pipeline (`src/transform.py`) :

- retirer les paramètres de suivi des URL : `nettoie_url()`
- supprimer les doublons sur l'URL nettoyée : `supprime_doublons()`
- uniformiser les apostrophes et retirer les "..." de fin : `nettoie_texte()`
- convertir les dates et le rang de fiabilité : `convertit_types()`
- vérifier que chaque image s'ouvre (colonne `a_image`) : `valide_image()`
- signaler les images partagées entre plusieurs articles : `verifie_association()`
- remplacer l'auteur et le label manquants : `remplit_manquants()`